In [13]:
import pandas as pd
import numpy as np
import plotly.express as px
import matplotlib.pyplot as plt
import kaleido 
import seaborn as sns
import warnings 
warnings.filterwarnings('ignore')
%matplotlib inline
pd.set_option('display.max_columns', None)
pd.options.display.float_format = '{:.2f}'.format

In [3]:
#na_values=[''] ona görə lazımdı ki TERRİTORY sütununda NA dəyərini Pandas NaN kimi qəbul etməsin
df = pd.read_csv(r'C:\Users\rashid\Desktop\rfm_segmentation\data\processed\sales_data_cleaned.csv', parse_dates=['ORDERDATE'], na_values=[''], keep_default_na=False)
df.head()

,ORDERNUMBER,QUANTITYORDERED,PRICEEACH,ORDERLINENUMBER,SALES,ORDERDATE,STATUS,QTR_ID,MONTH_ID,YEAR_ID,PRODUCTLINE,MSRP,PRODUCTCODE,CUSTOMERNAME,PHONE,ADDRESSLINE1,ADDRESSLINE2,CITY,STATE,POSTALCODE,COUNTRY,TERRITORY,CONTACTLASTNAME,CONTACTFIRSTNAME,DEALSIZE
0,10107,30,95.70,2,2871.00,2003-02-24,Shipped,1,2,2003,Motorcycles,95,S10_1678,Land of Toys Inc.,2125557818,897 Long Airport Avenue,Not Provided,NYC,NY,10022,USA,NA,Yu,Kwai,Small
1,10121,34,81.35,5,2765.90,2003-05-07,Shipped,2,5,2003,Motorcycles,95,S10_1678,Reims Collectables,26.47.1555,59 rue de l'Abbaye,Not Provided,Reims,Not Applicable,51100,France,EMEA,Henriot,Paul,Small
2,10134,41,94.74,2,3884.34,2003-07-01,Shipped,3,7,2003,Motorcycles,95,S10_1678,Lyon Souveniers,+33 1 46 62 7555,27 rue du Colonel Pierre Avia,Not Provided,Paris,Not Applicable,75508,France,EMEA,Da Cunha,Daniel,Medium
3,10145,45,83.26,6,3746.70,2003-08-25,Shipped,3,8,2003,Motorcycles,95,S10_1678,Toys4GrownUps.com,6265557265,78934 Hillside Dr.,Not Provided,Pasadena,CA,90003,USA,NA,Young,Julie,Medium
4,10159,49,100.00,14,5205.27,2003-10-10,Shipped,4,10,2003,Motorcycles,95,S10_1678,Corporate Gift Ideas Co.,6505551386,7734 Strong St.,Not Provided,San Francisco,CA,94217,USA,NA,Brown,Julie,Medium


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2823 entries, 0 to 2822
Data columns (total 25 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   ORDERNUMBER       2823 non-null   int64         
 1   QUANTITYORDERED   2823 non-null   int64         
 2   PRICEEACH         2823 non-null   float64       
 3   ORDERLINENUMBER   2823 non-null   int64         
 4   SALES             2823 non-null   float64       
 5   ORDERDATE         2823 non-null   datetime64[us]
 6   STATUS            2823 non-null   str           
 7   QTR_ID            2823 non-null   int64         
 8   MONTH_ID          2823 non-null   int64         
 9   YEAR_ID           2823 non-null   int64         
 10  PRODUCTLINE       2823 non-null   str           
 11  MSRP              2823 non-null   int64         
 12  PRODUCTCODE       2823 non-null   str           
 13  CUSTOMERNAME      2823 non-null   str           
 14  PHONE             2823 non-null   s

In [5]:
reference_date = df['ORDERDATE'].max() + pd.Timedelta(days=1)
reference_date

Timestamp('2005-06-01 00:00:00')

In [6]:
rfm = df.groupby('CUSTOMERNAME').agg(
    Recency=('ORDERDATE', lambda x: (reference_date - x.max()).days),
    Frequency=('ORDERNUMBER', 'nunique'),
    Monetary=('SALES', 'sum')
).reset_index()

print(rfm.shape)
rfm.describe()

(92, 4)


,Recency,Frequency,Monetary
count,92.00,92.00,92.00
mean,182.83,3.34,109050.31
std,131.42,2.92,110308.61
min,1.00,1.00,9129.35
25%,81.25,2.00,70129.43
50%,186.00,3.00,86522.61
75%,230.25,3.00,120575.88
max,509.00,26.00,912294.11


In [7]:
rfm['R_score'] = pd.qcut(rfm['Recency'], q=4, labels=[4, 3, 2, 1]).astype(int)

rfm['F_score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=4, labels=[1, 2, 3, 4]).astype(int)

rfm['M_score'] = pd.qcut(rfm['Monetary'], q=4, labels=[1, 2, 3, 4]).astype(int)

rfm['RFM_Segment'] = (
    rfm['R_score'].astype(str) +
    rfm['F_score'].astype(str) +
    rfm['M_score'].astype(str)
)

rfm['RFM_Score'] = rfm[['R_score', 'F_score', 'M_score']].sum(axis=1)

In [8]:
def assign_segment(row):
    r, f, m = row['R_score'], row['F_score'], row['M_score']
    
    if r == 4 and f == 4 and m == 4:
        return 'Champions'
    elif r >= 3 and f >= 3 and m >= 3:
        return 'Loyal'
    elif r == 1 and f >= 3 and m >= 3:
        return 'At Risk'
    elif r >= 2 and f <= 2 and m <= 2:
        return 'Needs Attention'
    elif r == 1 and f == 1 and m == 1:
        return 'Lost'

rfm['Segment'] = rfm.apply(assign_segment, axis=1)

In [9]:
segment_summary = rfm.groupby('Segment').agg(
    Customer_Count=('CUSTOMERNAME', 'count'),
    Avg_Recency=('Recency', 'mean'),
    Avg_Frequency=('Frequency', 'mean'),
    Avg_Monetary=('Monetary', 'mean')
).round(1).sort_values('Customer_Count', ascending=False)

print(segment_summary)

                 Customer_Count  Avg_Recency  Avg_Frequency  Avg_Monetary
Segment                                                                  
Loyal                        17       110.30           3.60     132894.90
Needs Attention              16       156.80           2.60      60769.70
Lost                         11       360.50           1.90      52367.00
Champions                     9        19.40           8.10     290097.70
At Risk                       1       456.00           3.00     142874.20


In [10]:
final_table = segment_summary.reset_index()[
    ['Segment', 'Customer_Count', 'Avg_Recency', 'Avg_Frequency', 'Avg_Monetary']
]
final_table

,Segment,Customer_Count,Avg_Recency,Avg_Frequency,Avg_Monetary
0,Loyal,17,110.30,3.60,132894.90
1,Needs Attention,16,156.80,2.60,60769.70
2,Lost,11,360.50,1.90,52367.00
3,Champions,9,19.40,8.10,290097.70
4,At Risk,1,456.00,3.00,142874.20


In [15]:
import plotly.express as px

fig = px.scatter_3d(
    rfm,
    x='Recency',
    y='Frequency',
    z='Monetary',
    color='Segment',
    hover_name='CUSTOMERNAME',
    hover_data=['R_score', 'F_score', 'M_score', 'RFM_Segment'],
    title='Müştəri Seqmentləri: RFM 3D İnteraktiv Görünüş',
    opacity=0.8
)

fig.update_layout(
    scene=dict(
        xaxis_title='Recency (gün)',
        yaxis_title='Frequency (sifariş)',
        zaxis_title='Monetary ($)'
    ),
    legend_title='Seqment'
)

fig.write_html("rfm_3d_plot.html")

## 📊 RFM Seqmentləri üzrə Marketinq Strategiyaları

---

### 1. Champions (Çempionlar)
* **Status:** Ən aktiv, ən çox və tez-tez alver edən VIP müştərilər.
* **Təklif olunan addımlar:**
  * **VIP Proqramı:** Müştərini xüsusi VIP proqramına daxil edin və fərdi imtiyazlar təmin edin.
  * **Həftəlik Həvəsləndirmə:** Hər həftə etdiyi alış-verişlər üçün fərdi **endirim kuponları** təqdim edin.

---

### 2. Loyal (Sadiq Müştərilər)
* **Status:** Mütəmadi olaraq bizi seçən və yüksək sadiqlik göstərən müştərilər.
* **Təklif olunan addımlar:**
  * **Loyalty Program:** Sadiqlik (bonuslar/xallar) proqramına cəlb edin.
  * **Cross-sell & Up-sell:** Keçmiş alışlarına uyğun tamamlayıcı məhsullar (cross-sell) və ya daha üst model məhsullar (up-sell) təklif edən kampaniyalar keçirin.

---

### 3. At Risk (Risk Altında Olanlar)
* **Status:** Əvvəllər çox aktiv və dəyərli olan, lakin son zamanlar uzaqlaşmış müştərilər.
* **Təklif olunan addımlar:**
  * **Klassik Vərdişlərə Uyğun Endirimlər:** Əvvəllər aldığı konkret produktlara görə xüsusi endirim kampaniyaları tərtib edin.
  * **Birbaşa Əlaqə:** Email vasitəsilə bildirişlər göndərin və ya **şəxsi zənglər** edərək səbəbini öyrənib yenidən qazanın.

---

### 4. Needs Attention (Diqqət Tələb Edənlər)
* **Status:** Aşağı dəyərli, lakin hazırda aktiv olan müştəri seqmenti.
* **Təklif olunan addımlar:**
  * **Engagement (Cəlbetmə):** Aktivliyi və marağı artırmaq üçün maraq oyandıran kontentlər paylaşın.
  * **Satış Təkanları:** Kiçik həcmli endirimlər və məhsul dəstləri (**bundle təklifləri**) ilə ortalama çeki böyüdün.

---

### 5. Lost (İtirilmiş Müştərilər)
* **Status:** Uzun müddətdir alver etməyən və az dəyər gətirmiş müştərilər.
* **Təklif olunan addımlar:**
  * **Aşağı Büdcəli Reaktivasiya:** Yüksək xərc çəkmədən minimal resurslarla (avtomatik SMS/Email) reaktivasiya kampaniyası aparın.
  * **Güclü Təklif:** Minimal xərclə maksimal diqqət çəkən "Geri dön" təklifləri/böyük bir dəfəlik endirimlər təqdim edin.